## Cellpose-SAM on Synthetic Data


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

### Init


In [ ]:
# !uv pip install git+https://www.github.com/mouseland/cellpose.git
# !uv pip install git+https://github.com/facebookresearch/dinov3.git

In [ ]:
import os
import sys
import zarr
import glob
import json
import warnings
import numpy as np
import polars as pl
import pandas as pd
import tracksdata as td
import matplotlib.pyplot as plt

from scipy import ndimage
from pathlib import Path
from tqdm import tqdm

sys.path.insert(0, '../src')

from cellpose import models, core, io, plot

In [ ]:
from params import *
from utils.plot import *

In [ ]:
FOLDER = '../notebooks/biohub-synthetic-dataset/working/synth_out_3/biohub_synthetic/static/'

In [ ]:
print(f'Found {len(os.listdir(FOLDER))} volumes')

### Model

In [ ]:
model = models.CellposeModel(gpu=core.use_gpu())
print('GPU enabled:', core.use_gpu())

In [ ]:
flow_threshold = 1
cellprob_threshold = -1
tile_norm_blocksize = 0

### Main

In [ ]:
PLOT = False
CELLPOSE = True

SAVE_FOLDER = '../output/segs_3/'
os.makedirs(SAVE_FOLDER, exist_ok=True)

In [ ]:
for sample_name in tqdm(os.listdir(FOLDER)):
    if os.path.exists(f'{SAVE_FOLDER}/{sample_name}'):
        continue

    # Load data
    npy = np.load(os.path.join(FOLDER, sample_name))
    
    if "static" in FOLDER:
        img = npy['volume'][..., None]
        gt_masks = npy['mask']
        gt = np.round(npy['centroids'], 0).astype(np.int32)
    else:
        frame = 0

        img = npy['volumes'][frame, ..., None]

        gt = np.round(npy['nodes'], 0).astype(np.int32)
        gt = gt[gt[:, 0] == frame]
        gt = gt[:, [1, 2, 3]]

    if PLOT:
        z = np.random.randint(1, img.shape[0] - 1)
        plt.figure(figsize=(15, 5))

        for i in range(3):
            plt.subplot(1, 3, i + 1)

            z_ = z - 1 + i
            plt.title(f"z = {z_}")
            plt.imshow(img[z_], cmap="gray")

            gt_ = gt[np.abs(gt[:, 0] - z_) <= 3]
            plt.scatter(gt_[:, 2], gt_[:, 1], c='red', s=10)
            plt.axis('off')
        plt.show()

    if CELLPOSE:
        masks, flows, styles = model.eval(
            img,
            batch_size=64,
            flow_threshold=flow_threshold,
            cellprob_threshold=cellprob_threshold,
        )

        np.savez(
            f"{SAVE_FOLDER}/{sample_name}",
            img=img.squeeze(),
            masks=gt_masks,
            cellpose_masks=masks,
            flows=flows[0],  # XY flow in HSV 0-255
            probas=flows[2],  # cell proba logits
            # gradients=flows[3],  # gradients
            centroids=gt
        )
    else:
        np.savez(
            f"{SAVE_FOLDER}/{sample_name}",
            img=img.squeeze(),
            masks=gt_masks,
            centroids=gt
        )

    if PLOT and CELLPOSE:
        i = 32
        plt.figure(figsize=(12, 4))
        gt_f = gt[np.abs(gt[:, 0] - i) <= 3]
        show_cellpose_segmentation(img[i], masks[i], flows[0][i], nodes=gt_f)
        plt.tight_layout()
        plt.show()

        break

Done !